# Fine-Tuning Continuation – Pneumonia Detection Model

This notebook continues training (fine-tuning) from a previously saved ResNet50 pneumonia detection model.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [ ]:
from tensorflow.keras.models import load_model


MODEL_PATH = "/content/drive/MyDrive/pneumonia_project/pneumonia_resnet50_final.keras"

model = load_model(MODEL_PATH)
print("Model loaded successfully!")


Model loaded successfully!


In [ ]:
for layer in model.layers:
    print(layer.name)


input_layer
conv1_pad
conv1_conv
conv1_bn
conv1_relu
pool1_pad
pool1_pool
conv2_block1_1_conv
conv2_block1_1_bn
conv2_block1_1_relu
conv2_block1_2_conv
conv2_block1_2_bn
conv2_block1_2_relu
conv2_block1_0_conv
conv2_block1_3_conv
conv2_block1_0_bn
conv2_block1_3_bn
conv2_block1_add
conv2_block1_out
conv2_block2_1_conv
conv2_block2_1_bn
conv2_block2_1_relu
conv2_block2_2_conv
conv2_block2_2_bn
conv2_block2_2_relu
conv2_block2_3_conv
conv2_block2_3_bn
conv2_block2_add
conv2_block2_out
conv2_block3_1_conv
conv2_block3_1_bn
conv2_block3_1_relu
conv2_block3_2_conv
conv2_block3_2_bn
conv2_block3_2_relu
conv2_block3_3_conv
conv2_block3_3_bn
conv2_block3_add
conv2_block3_out
conv3_block1_1_conv
conv3_block1_1_bn
conv3_block1_1_relu
conv3_block1_2_conv
conv3_block1_2_bn
conv3_block1_2_relu
conv3_block1_0_conv
conv3_block1_3_conv
conv3_block1_0_bn
conv3_block1_3_bn
conv3_block1_add
conv3_block1_out
conv3_block2_1_conv
conv3_block2_1_bn
conv3_block2_1_relu
conv3_block2_2_conv
conv3_block2_2_bn
co

In [ ]:
BASE_DIR = "/content/drive/MyDrive/pneumonia_project/chest_xray"
train_dir = os.path.join(BASE_DIR, "train")
val_dir = os.path.join(BASE_DIR, "val")


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

DATA_PATH = "/content/drive/MyDrive/pneumonia_project/chest_xray"

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.1,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    DATA_PATH + "/train",
    target_size=(224,224),
    batch_size=32,
    class_mode="binary"
)

validation_generator = val_datagen.flow_from_directory(
    DATA_PATH + "/test",
    target_size=(224,224),
    batch_size=32,
    class_mode="binary",
    shuffle=False
)


Found 5216 images belonging to 2 classes.
Found 624 images belonging to 2 classes.


In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)

class_weight_dict = dict(enumerate(class_weights))
class_weight_dict


{0: np.float64(1.9448173005219984), 1: np.float64(0.6730322580645162)}

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model

base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(1, activation="sigmoid")(x)

model = Model(inputs=base_model.input, outputs=output)


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:
import os
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

SAVE_PATH = "/content/drive/MyDrive/pneumonia_project"
os.makedirs(SAVE_PATH, exist_ok=True)

checkpoint = ModelCheckpoint(
    filepath=SAVE_PATH + "/resnet50_best.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

callbacks = [checkpoint, early_stop]


In [ ]:
set_trainable = False

for layer in model.layers:
    if layer.name.startswith("conv5"):
        set_trainable = True
    if set_trainable:
        layer.trainable = True


In [ ]:
for layer in model.layers[-20:]:
    print(layer.name, layer.trainable)


conv5_block2_2_bn True
conv5_block2_2_relu True
conv5_block2_3_conv True
conv5_block2_3_bn True
conv5_block2_add True
conv5_block2_out True
conv5_block3_1_conv True
conv5_block3_1_bn True
conv5_block3_1_relu True
conv5_block3_2_conv True
conv5_block3_2_bn True
conv5_block3_2_relu True
conv5_block3_3_conv True
conv5_block3_3_bn True
conv5_block3_add True
conv5_block3_out True
global_average_pooling2d True
dense True
dropout True
dense_1 True


In [ ]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [ ]:
history_finetune = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=6,
    class_weight=class_weight_dict,
    callbacks=callbacks
)


Epoch 1/6
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - accuracy: 0.9396 - loss: 0.1618
Epoch 1: val_accuracy did not improve from 0.86699
163/163 ━━━━━━━━━━━━━━━━━━━━ 1540s 9s/step - accuracy: 0.9396 - loss: 0.1618 - val_accuracy: 0.8526 - val_loss: 0.3174
Epoch 2/6
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - accuracy: 0.9467 - loss: 0.1397
Epoch 2: val_accuracy did not improve from 0.86699
163/163 ━━━━━━━━━━━━━━━━━━━━ 1562s 10s/step - accuracy: 0.9467 - loss: 0.1397 - val_accuracy: 0.8542 - val_loss: 0.3041
Epoch 3/6
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - accuracy: 0.9499 - loss: 0.1287
Epoch 3: val_accuracy did not improve from 0.86699
163/163 ━━━━━━━━━━━━━━━━━━━━ 1525s 9s/step - accuracy: 0.9499 - loss: 0.1288 - val_accuracy: 0.7532 - val_loss: 0.7560
Epoch 4/6
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - accuracy: 0.9550 - loss: 0.1203
Epoch 4: val_accuracy did not improve from 0.86699
163/163 ━━━━━━━━━━━━━━━━━━━━ 1498s 9s/step - accuracy: 0.9550 - loss: 0.1203 - val_accuracy: 0.7147 - 

In [ ]:
history_finetune.history['accuracy']
history_finetune.history['val_accuracy']


[0.8525640964508057,
 0.8541666865348816,
 0.7532051205635071,
 0.7147436141967773,
 0.8028846383094788]

In [ ]:
model.save("/content/drive/MyDrive/pneumonia_project/pneumonia_resnet50_final_2.keras")
print("Model saved successfully.")


Model saved successfully.


In [ ]:
model.save("/content/drive/MyDrive/pneumonia_project/pneumonia_resnet50_final_3.keras")
print("Model saved successfully.")


Model saved successfully.


In [ ]:
model.save("/content/drive/MyDrive/pneumonia_project/pneumonia_resnet50_finetuned.keras")
